# Torch Adam - RMS Spot Size Optimization

- Note that optimization in PyTorch is fundamentally the same as the standard optimization using NumPy/SciPy. The only user-facing difference is the use of a torch-based optimizer.

In [ ]:
import optiland.backend as be
from optiland import optic, optimization
from optiland.optimization import minimize

be.set_backend("torch")  # Set the backend to PyTorch
be.grad_mode.enable()  # Enable gradient tracking


Define a starting lens:

In [ ]:
lens = optic.Optic()

# add surfaces
lens.surfaces.add(index=0, thickness=be.inf)
lens.surfaces.add(index=1, thickness=7, radius=1000, material="N-SF11", is_stop=True)
lens.surfaces.add(index=2, thickness=30, radius=-1000)
lens.surfaces.add(index=3)

# set aperture
lens.set_aperture(aperture_type="EPD", value=15)

# add field
lens.fields.set_type(field_type="angle")
lens.fields.add(y=0)

# add wavelength
lens.wavelengths.add(value=0.55, is_primary=True)

# draw lens
_ = lens.draw(num_rays=5)

Define optimization problem:

In [ ]:
problem = optimization.OptimizationProblem()

Add operands (targets for optimization):

In [ ]:
"""
Define RMS spot size properties for the optimization:
    1. Surface number = -1, implying last surface (image surface)
    2. Normalized field coordinates (Hx, Hy) = (0, 0)
    3. Number of rays = 5, corresponds to number of rings in hexapolar distribution
        (see distribution documentation)
    4. Wavelength = 0.55 µm
    5. Pupil distribution = hexapolar
"""

input_data = {
    "optic": lens,
    "surface_number": -1,
    "Hx": 0,
    "Hy": 0,
    "num_rays": 5,
    "wavelength": 0.55,
    "distribution": "hexapolar",
}

# add RMS spot size operand
problem.add_operand(
    operand_type="rms_spot_size",
    target=0,
    weight=1,
    input_data=input_data,
)

Define variables - let radius of curvature vary for both surfaces, at surface index 1 and 2:

In [ ]:
problem.add_variable(lens, "radius", surface_number=1)
problem.add_variable(lens, "radius", surface_number=2)

Check initial merit function value and system properties:

In [ ]:
problem.info()

Define optimizer:

Run optimization:

- Define learning rate (`lr`) and the learning rate decay factor (`gamma`)

In [ ]:
result = minimize(problem, "adam", maxiter=250, lr=0.1, gamma=0.99, disp=False)
print(result)


In [ ]:
be.set_backend("numpy")  # Reset backend to NumPy


Print merit function value and system properties after optimization:

In [ ]:
problem.info()

Draw final lens:

In [ ]:
_ = lens.draw(num_rays=5)